In [ ]:
REPO = "/kaggle/working/ComputerVisionAndImageProcessing"
!git clone -q https://github.com/nataliachoszczyk/ComputerVisionAndImageProcessing.git {REPO} 2>/dev/null || git -C {REPO} pull
%cd {REPO}/01-image-modification

In [ ]:
from pathlib import Path

if not Path("/tmp/div2k").exists():
    !wget -q --show-progress -P /tmp http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip
    !wget -q --show-progress -P /tmp http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip
    !unzip -q /tmp/DIV2K_train_HR.zip -d /tmp/div2k
    !unzip -q /tmp/DIV2K_valid_HR.zip -d /tmp/div2k
    !rm /tmp/DIV2K_*.zip

print(len(list(Path("/tmp/div2k").rglob("*.png"))))

In [ ]:
from data import cut_patches
from sr_data_preparation import make_lr_patches

PATCHES = "/tmp/patches"

if not Path(PATCHES).exists():
    cut_patches("/tmp/div2k", PATCHES)
    make_lr_patches(PATCHES)

In [ ]:
for split in ["train", "val", "test"]:
    print(split, {d.name: len(list(d.glob("*.png"))) for d in sorted(Path(PATCHES, split).iterdir())})

In [ ]:
import cv2
import matplotlib.pyplot as plt

name = sorted(Path(PATCHES, "test/original").glob("*.png"))[80].name
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, folder in zip(axes, ["original", "lr_x4", "lr_x8"]):
    img = cv2.cvtColor(cv2.imread(f"{PATCHES}/test/{folder}/{name}"), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(f"{folder} {img.shape[1]}×{img.shape[0]}")
    ax.axis("off")

In [ ]:
!pip install -q lpips

In [ ]:
import shutil
import numpy as np
import pandas as pd
import torch

from data import PatchDataset, PATCH_SIZE
from metrics import evaluate

WORK = Path("/kaggle/working")
device = "cuda" if torch.cuda.is_available() else "cpu"

for f in Path("/kaggle/input").rglob("*"):
    if f.name.startswith(("unet_", "history_", "bicubic_x")):
        shutil.copy(f, WORK)


def bicubic(lr):
    imgs = lr.cpu().permute(0, 2, 3, 1).numpy()
    up = [cv2.resize(img, (PATCH_SIZE, PATCH_SIZE), interpolation=cv2.INTER_CUBIC) for img in imgs]
    return torch.from_numpy(np.stack(up)).permute(0, 3, 1, 2).clamp(0, 1)


results = {}
for scale in [4, 8]:
    csv = WORK / f"bicubic_x{scale}.csv"
    if not csv.exists():
        test = PatchDataset(f"{PATCHES}/test/lr_x{scale}", f"{PATCHES}/test/original")
        evaluate(bicubic, test, device).to_csv(csv)
    results[f"bicubic_x{scale}"] = pd.read_csv(csv, index_col=0)

pd.DataFrame({name: df.mean() for name, df in results.items()}).T

In [ ]:
from model import UNet

for upsample in ["bicubic", "learned"]:
    model = UNet(scale=4, upsample=upsample)
    print(upsample, model(torch.rand(2, 3, 64, 64)).shape, sum(p.numel() for p in model.parameters()))

In [ ]:
from training import train

VARIANTS = {"": "bicubic", "learned_": "learned"}

for prefix, upsample in VARIANTS.items():
    for scale in [4, 8]:
        checkpoint = WORK / f"unet_{prefix}x{scale}.pt"
        if checkpoint.exists():
            continue
        train_set = PatchDataset(f"{PATCHES}/train/lr_x{scale}", f"{PATCHES}/train/original")
        val_set = PatchDataset(f"{PATCHES}/val/lr_x{scale}", f"{PATCHES}/val/original")
        history = train(UNet(scale=scale, upsample=upsample), train_set, val_set, checkpoint, epochs=25)
        history.to_csv(WORK / f"history_{prefix}x{scale}.csv", index=False)

In [ ]:
for prefix, upsample in VARIANTS.items():
    for scale in [4, 8]:
        model = UNet(scale=scale, upsample=upsample)
        model.load_state_dict(torch.load(WORK / f"unet_{prefix}x{scale}.pt", map_location=device))
        model.to(device).eval()
        test = PatchDataset(f"{PATCHES}/test/lr_x{scale}", f"{PATCHES}/test/original")
        results[f"unet_{prefix}x{scale}"] = evaluate(model, test, device)

order = [f"{m}_x{s}" for s in [4, 8] for m in ["bicubic", "unet", "unet_learned"]]
table = pd.DataFrame({name: df.mean() for name, df in results.items()}).T.loc[order]
table.to_csv(WORK / "results.csv")
table.round(4)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for prefix, upsample in VARIANTS.items():
    for scale in [4, 8]:
        h = pd.read_csv(WORK / f"history_{prefix}x{scale}.csv")
        label = f"x{scale} {upsample}"
        axes[0].plot(h.epoch, h.train_loss, label=label)
        axes[1].plot(h.epoch, h.val_psnr, label=label)
axes[0].set(title="Training loss (L1)", xlabel="epoch")
axes[1].set(title="Validation PSNR [dB]", xlabel="epoch")
for ax in axes:
    ax.legend()
fig.savefig(WORK / "learning_curves.png", dpi=150, bbox_inches="tight")

In [ ]:
from data import load_image
from metrics import psnr

names = sorted(p.name for p in Path(PATCHES, "test/original").glob("*.png"))[::900]

for scale in [4, 8]:
    models = {}
    for prefix, upsample in VARIANTS.items():
        m = UNet(scale=scale, upsample=upsample)
        m.load_state_dict(torch.load(WORK / f"unet_{prefix}x{scale}.pt", map_location="cpu"))
        models[upsample] = m.eval()

    fig, axes = plt.subplots(len(names), 5, figsize=(20, 4 * len(names)))
    for row, name in zip(axes, names):
        lr = load_image(f"{PATCHES}/test/lr_x{scale}/{name}")[None]
        hr = load_image(f"{PATCHES}/test/original/{name}")[None]
        with torch.no_grad():
            images = {
                f"input {256 // scale}×{256 // scale}": lr,
                "bicubic": bicubic(lr),
                "UNet (bicubic input)": models["bicubic"](lr).clamp(0, 1),
                "UNet (learned upsampling)": models["learned"](lr).clamp(0, 1),
                "original": hr,
            }
        for ax, (title, img) in zip(row, images.items()):
            if title != "original" and not title.startswith("input"):
                title += f" ({psnr(img, hr).item():.2f} dB)"
            ax.imshow(img[0].permute(1, 2, 0).numpy(), interpolation="nearest")
            ax.set_title(title)
            ax.axis("off")
    fig.suptitle(f"Super-resolution x{scale}", fontsize=16)
    fig.savefig(WORK / f"comparison_x{scale}.png", dpi=100, bbox_inches="tight")